# R10-H95 - text beats structure at this scale (the honest null)

Benchmark document set: the same interim identity pair set (252 pairs). Structure embeddings from node2vec over the **valid graph** - the 3818 typed relationship edges EXCLUDING `SIMILAR_TO` (a similarity-derived edge that would leak text signal) and `SAME_AS` (the identity edge we are predicting - direct leakage). 2520 nodes covered.

- **Hypothesis (null-leaning)** - node2vec structural embeddings rank the labeled duplicate pairs WORSE than Titan text by > 0.05 AUC
- **Constructive refuter** - text + structure concatenation beats text alone by > 0.05 AUC (structure carries complementary identity signal)
- **Metrics** - overall AUC (dup vs hard-neg = sibling + false_merge) and variance-vs-sibling AUC

In [1]:
# No GPU needed - node2vec + gensim run on CPU
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""


In [2]:
# Imports
import datetime, glob, json
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from rich import print as rprint
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
from neo4j import GraphDatabase
np.random.seed(42)


/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Configuration
NEO4J_URI = "bolt://user-konrad.jelen-kgf-neo4j2:7687"  # READ-ONLY
NEO4J_AUTH = ("neo4j", "kgfoundry")
EXCLUDE_RELS = {"SIMILAR_TO", "SAME_AS"}  # leakage: derived-similarity and the identity edge itself
N2V_DIM = 128
N2V_WALK_LEN = 30
N2V_NUM_WALKS = 200
N2V_P, N2V_Q = 1.0, 1.0
BAR_NULL = 0.05   # text beats structure by this -> null holds
BAR_REFUTE = 0.05 # concat beats text by this -> refuter fires
OVERALL_POS = {"variance", "resolver_miss", "samename"}
OVERALL_NEG = {"sibling", "false_merge"}
rprint(f"[bold cyan]H95 config[/bold cyan]  node2vec dim={N2V_DIM} walks={N2V_NUM_WALKS} len={N2V_WALK_LEN}  exclude={EXCLUDE_RELS}")


H95 config  node2vec dim=128 walks=200 len=30  exclude={'SIMILAR_TO', 'SAME_AS'}

## Load valid graph + frozen pairs + Titan vectors

In [4]:
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH, notifications_min_severity="OFF")
with driver.session() as s:
    edges = s.run(
        "MATCH (a:Entity)-[r]->(b:Entity) WHERE NOT type(r) IN $ex "
        "RETURN a.id AS a, b.id AS b", ex=list(EXCLUDE_RELS)).data()
    ents = s.run("MATCH (e:Entity) WHERE e.embedding IS NOT NULL "
                 "RETURN e.id AS id, e.name AS name, e.embedding AS emb").data()
driver.close()

name_to_id = {}
for r in ents:
    name_to_id.setdefault(r["name"], r["id"])
TITAN = {r["id"]: np.asarray(r["emb"], dtype=np.float32) for r in ents}

G = nx.Graph()
for e in edges:
    G.add_edge(str(e["a"]), str(e["b"]))
print("valid graph:", G.number_of_nodes(), "nodes", G.number_of_edges(), "edges")

rep_path = sorted(glob.glob("../reports/matching-r12-foundation-*.json"))[-1]
PAIRS_RAW = json.load(open(rep_path))["pairs"]
P = []
for p in PAIRS_RAW:
    a, b = name_to_id.get(p["a"]), name_to_id.get(p["b"])
    if a and b:
        P.append(dict(a=a, b=b, y=p["y"], cls=p["cls"]))
print("aligned pairs:", len(P), "| source:", rep_path)


valid graph: 2520 nodes 3634 edges
aligned pairs: 252 | source: ../reports/matching-r12-foundation-20260706-211008.json


## Fit node2vec

In [5]:
n2v = Node2Vec(G, dimensions=N2V_DIM, walk_length=N2V_WALK_LEN, num_walks=N2V_NUM_WALKS,
               p=N2V_P, q=N2V_Q, workers=4, seed=42, quiet=True)
model = n2v.fit(window=10, min_count=1, batch_words=256, seed=42)
N2V = {k: np.asarray(model.wv[k], dtype=np.float32) for k in model.wv.index_to_key}
print("node2vec vectors:", len(N2V), "dim", N2V_DIM)

# pair coverage in the structural space
cov = sum(1 for p in P if str(p["a"]) in N2V and str(p["b"]) in N2V)
print(f"pairs with both nodes embedded: {cov}/{len(P)}")


node2vec vectors: 2520 dim 128
pairs with both nodes embedded: 208/252


In [6]:
# Score pairs: titan cosine, node2vec cosine (0 if a node is missing), plus feature stack
def cos(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12))

titan_s, n2v_s, concat_s = [], [], []
for p in P:
    a, b = p["a"], p["b"]
    titan_s.append(cos(TITAN[a], TITAN[b]))
    sa, sb = str(a), str(b)
    if sa in N2V and sb in N2V:
        n2v_s.append(cos(N2V[sa], N2V[sb]))
        # concat vector cosine: L2-normalize each block, then cosine of the stack
        ta = TITAN[a] / (np.linalg.norm(TITAN[a]) + 1e-12)
        tb = TITAN[b] / (np.linalg.norm(TITAN[b]) + 1e-12)
        na = N2V[sa] / (np.linalg.norm(N2V[sa]) + 1e-12)
        nb = N2V[sb] / (np.linalg.norm(N2V[sb]) + 1e-12)
        concat_s.append(cos(np.concatenate([ta, na]), np.concatenate([tb, nb])))
    else:
        n2v_s.append(0.0)
        concat_s.append(titan_s[-1])  # no structure -> fall back to text
titan_s, n2v_s, concat_s = map(np.array, (titan_s, n2v_s, concat_s))


In [7]:
# Metrics
def subset_auc(scores, pos_cls, neg_cls):
    y, s = [], []
    for i, p in enumerate(P):
        if p["cls"] in pos_cls: y.append(1); s.append(scores[i])
        elif p["cls"] in neg_cls: y.append(0); s.append(scores[i])
    return float(roc_auc_score(y, s))
def both(scores):
    return subset_auc(scores, OVERALL_POS, OVERALL_NEG), subset_auc(scores, {"variance"}, {"sibling"})

t_o, t_v = both(titan_s)
n_o, n_v = both(n2v_s)
c_o, c_v = both(concat_s)

# honest 2-feature logistic (titan_cos, n2v_cos) with 5-fold CV on the scored subset
sub, ysub = [], []
for i, p in enumerate(P):
    if p["cls"] in OVERALL_POS: sub.append(i); ysub.append(1)
    elif p["cls"] in OVERALL_NEG: sub.append(i); ysub.append(0)
sub, ysub = np.array(sub), np.array(ysub)
Xtn = np.column_stack([titan_s, n2v_s])[sub]
cv = StratifiedKFold(5, shuffle=True, random_state=42)
p_fuse = cross_val_predict(LogisticRegression(max_iter=1000), Xtn, ysub, cv=cv, method="predict_proba")[:, 1]
p_txt = cross_val_predict(LogisticRegression(max_iter=1000), titan_s[sub].reshape(-1,1), ysub, cv=cv, method="predict_proba")[:, 1]
fuse_o = float(roc_auc_score(ysub, p_fuse))
txt_o = float(roc_auc_score(ysub, p_txt))

rprint("[bold cyan]H95 results (overall / var-vs-sib)[/bold cyan]")
rprint(f"  titan text    overall [yellow]{t_o:.4f}[/yellow]  var-vs-sib [yellow]{t_v:.4f}[/yellow]")
rprint(f"  node2vec      overall [yellow]{n_o:.4f}[/yellow]  var-vs-sib [yellow]{n_v:.4f}[/yellow]")
rprint(f"  concat cosine overall [yellow]{c_o:.4f}[/yellow]  var-vs-sib [yellow]{c_v:.4f}[/yellow]")
rprint(f"  logistic: text-only {txt_o:.4f}  ->  text+structure {fuse_o:.4f}  gain {fuse_o-txt_o:+.4f}")

null_holds = (t_o - n_o) > BAR_NULL
refuter = ((c_o - t_o) > BAR_REFUTE) or ((fuse_o - txt_o) > BAR_REFUTE)
rprint(f"  text beats structure by {t_o-n_o:+.4f} (null holds if > {BAR_NULL}): [green]{null_holds}[/green]")
rprint(f"  concat/fusion beats text by max({c_o-t_o:+.4f}, {fuse_o-txt_o:+.4f}) (refuter if > {BAR_REFUTE}): [red]{refuter}[/red]")
verdict = "REFUTED-null(fusion wins)" if refuter else ("CONFIRMED-null(text wins)" if null_holds else "INCONCLUSIVE(text wins but < 0.05 margin)")
rprint(f"  [bold]H95 verdict: {verdict}[/bold]")


H95 results (overall / var-vs-sib)

titan text    overall 0.8931  var-vs-sib 0.9713

node2vec      overall 0.3645  var-vs-sib 0.4532

concat cosine overall 0.5766  var-vs-sib 0.7883

logistic: text-only 0.8895  ->  text+structure 0.8501  gain -0.0394

text beats structure by +0.5286 (null holds if > 0.05): True

concat/fusion beats text by max(-0.3165, -0.0394) (refuter if > 0.05): False

H95 verdict: CONFIRMED-null(text wins)

In [8]:
# Save report
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = dict(
    pair_set=rep_path, n_pairs=len(P),
    valid_graph=dict(nodes=G.number_of_nodes(), edges=G.number_of_edges(), excluded=list(EXCLUDE_RELS)),
    node2vec=dict(dim=N2V_DIM, walk_len=N2V_WALK_LEN, num_walks=N2V_NUM_WALKS, coverage=int(cov)),
    titan=dict(overall=t_o, varsib=t_v),
    structure=dict(overall=n_o, varsib=n_v),
    concat_cosine=dict(overall=c_o, varsib=c_v),
    logistic_fusion=dict(text_only=txt_o, text_plus_structure=fuse_o, gain=fuse_o-txt_o),
    null_holds=bool(null_holds), refuter_fires=bool(refuter), verdict=verdict,
)
outp = f"../reports/structure-embed-r10-h95-{stamp}.json"
json.dump(out, open(outp, "w"), indent=1, default=float)
print("wrote", outp)


wrote ../reports/structure-embed-r10-h95-20260707-091458.json
